# 09 — M4: Forecast de Clicks

Predicción del volumen de clicks semanal para las próximas semanas.
Permite al equipo de marketing anticipar la demanda y ajustar la intensidad de las campañas.

**¿Qué es un forecast de series temporales?**  
Una serie temporal es una secuencia de datos ordenados en el tiempo (aquí, clicks por semana).
El forecast consiste en usar los datos históricos para predecir valores futuros.

**Datos disponibles:** 53 semanas (enero 2025 – enero 2026), 11,940 clicks totales.  
**Modelo:** Prophet (Facebook/Meta) — maneja tendencia + estacionalidad de forma automática.  
**Outputs:** `forecast_clicks.csv` con predicciones semanales + intervalos de confianza.

In [ ]:
import warnings
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Anade src/ al path y reutiliza el helper compartido (mismo patron que 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

ROOT_PATH      = find_project_root()
DATA_PATH      = ROOT_PATH / 'data'
PROCESSED_PATH = DATA_PATH / 'processed'

np.random.seed(42)
print('ROOT:', ROOT_PATH)

# Semanas a predecir hacia adelante
FORECAST_WEEKS = 8

## 1 · Carga y agregación de datos

In [ ]:
events = pd.read_csv(PROCESSED_PATH / 'events.csv')
events['timestamp'] = pd.to_datetime(events['timestamp'])

print(f'Eventos totales:  {len(events):,}')
print(f'Rango temporal:   {events["timestamp"].min().date()} → {events["timestamp"].max().date()}')
print(f'Tipos de evento:\n{events["event_type"].value_counts().to_string()}')

In [ ]:
# Separar clicks de opens
clicks = events[events['event_type'] == 'click'].copy()
opens  = events[events['event_type'] == 'open'].copy()

# Agregación semanal (lunes como inicio de semana)
weekly_clicks = (
    clicks.set_index('timestamp')
    .resample('W-MON')['id_event']
    .count()
    .rename('clicks')
    .reset_index()
    .rename(columns={'timestamp': 'week'})
)
weekly_opens = (
    opens.set_index('timestamp')
    .resample('W-MON')['id_event']
    .count()
    .rename('opens')
    .reset_index()
    .rename(columns={'timestamp': 'week'})
)
weekly = weekly_clicks.merge(weekly_opens, on='week', how='outer').fillna(0).astype({'clicks': int, 'opens': int})
weekly['ctr'] = weekly['clicks'] / (weekly['clicks'] + weekly['opens']).replace(0, np.nan)

# Eliminar última semana si está incompleta (menos del 50% de la media)
media_clicks = weekly['clicks'].iloc[:-1].mean()
if weekly['clicks'].iloc[-1] < media_clicks * 0.5:
    weekly = weekly.iloc[:-1]
    print('Última semana eliminada por estar incompleta.')

print(f'\nSemanas disponibles: {len(weekly)}')
print(f'Clicks semanales — min:{weekly["clicks"].min()}  max:{weekly["clicks"].max()}  media:{weekly["clicks"].mean():.0f}')
display(weekly.head(3))
print('...')
display(weekly.tail(3))

## 2 · Análisis exploratorio de la serie temporal

Antes de modelar, visualizamos la serie para entender su comportamiento:
¿hay tendencia creciente/decreciente? ¿hay estacionalidad (patrones que se repiten)? ¿hay outliers?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(weekly['week'], weekly['clicks'], color='steelblue', linewidth=1.5, marker='o', markersize=3)
axes[0].fill_between(weekly['week'], weekly['clicks'], alpha=0.15, color='steelblue')
axes[0].set_ylabel('Clicks / semana')
axes[0].set_title('Volumen semanal de clicks')

axes[1].plot(weekly['week'], weekly['opens'], color='orange', linewidth=1.5, marker='o', markersize=3)
axes[1].fill_between(weekly['week'], weekly['opens'], alpha=0.15, color='orange')
axes[1].set_ylabel('Opens / semana')
axes[1].set_title('Volumen semanal de opens')

axes[2].plot(weekly['week'], weekly['ctr'] * 100, color='seagreen', linewidth=1.5, marker='o', markersize=3)
axes[2].set_ylabel('CTR (%)')
axes[2].set_title('Click-Through Rate semanal')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Serie temporal de campañas de email marketing (2025)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Descomposición de la serie: tendencia + estacionalidad + residuo
# Necesitamos al menos 2 períodos para la estacionalidad; usamos period=4 (mensual aprox.)
serie = weekly.set_index('week')['clicks'].asfreq('W-MON').fillna(0)

decomp = seasonal_decompose(serie, model='additive', period=4, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

decomp.observed.plot(ax=axes[0], color='steelblue'); axes[0].set_title('Serie original'); axes[0].set_ylabel('Clicks')
decomp.trend.plot(ax=axes[1], color='tomato');       axes[1].set_title('Tendencia'); axes[1].set_ylabel('Clicks')
decomp.seasonal.plot(ax=axes[2], color='seagreen');  axes[2].set_title('Estacionalidad (período=4 semanas)'); axes[2].set_ylabel('Clicks')
decomp.resid.plot(ax=axes[3], color='gray');         axes[3].set_title('Residuo'); axes[3].set_ylabel('Clicks')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Descomposición de la serie temporal de clicks', fontsize=13)
plt.tight_layout()
plt.show()

## 3 · Clicks semanales por sector

Desagregamos los clicks por sector para ver qué categorías tienen más actividad
y si hay diferencias de estacionalidad entre ellas.

In [ ]:
# Unir events con propensity_scores para tener el sector
prop = pd.read_csv(PROCESSED_PATH / 'propensity_scores.csv')
prop = prop.merge(events[['id_event', 'timestamp']], on='id_event', how='left')
prop['timestamp'] = pd.to_datetime(prop['timestamp'])

# Derivar semana (lunes) consistente con resample('W-MON')
prop['week'] = prop['timestamp'].dt.normalize() - pd.to_timedelta(prop['timestamp'].dt.weekday, unit='D')

click_sector = (
    prop[prop['target'] == 1]
    .groupby(['week', 'sector'])['id_event']
    .count()
    .unstack(fill_value=0)
)

top_sectors = prop[prop['target'] == 1]['sector'].value_counts().head(6).index

fig, ax = plt.subplots(figsize=(14, 5))
colors = sns.color_palette('tab10', len(top_sectors))
for sector, color in zip(top_sectors, colors):
    if sector in click_sector.columns:
        ax.plot(click_sector.index, click_sector[sector], label=sector, color=color, linewidth=1.5, marker='o', markersize=2)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.set_ylabel('Clicks / semana')
ax.set_title('Clicks semanales por sector (top 6)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print('\nTotal clicks por sector:')
print(prop[prop['target'] == 1]['sector'].value_counts().to_string())

## 4 · Modelo Prophet — Diagnóstico y filtrado del período activo

**¿Por qué el modelo fallaba con todos los datos?**

El dataset incluye ~30 semanas iniciales (enero–julio 2025) con 0–50 clicks por semana:
la campaña aún no había arrancado. Al incluirlas, Prophet aprende que "a principio de año hay 0 clicks"
y extrapola esa tendencia a enero 2026 → predicción = 0.

**Solución: filtrar al período activo.**
Usamos solo las semanas con ≥ 50 clicks (primera semana activa ≈ final de julio 2025).
Esto le da al modelo una señal limpia sobre el comportamiento real de la campaña.

**¿Por qué desactivamos estacionalidades?**
- `weekly_seasonality=False`: trabajamos con datos ya agregados por semana, no por día.
  No hay variación intrasemanal que modelar.
- `yearly_seasonality=False`: con menos de 6 meses de datos activos no podemos estimar
  un patrón anual (necesitaría al menos 2 años completos).

**`changepoint_prior_scale=0.5`** (más alto que el default 0.05): le da flexibilidad al modelo
para capturar tanto el crecimiento rápido (jul–oct) como la caída posterior (nov–dic).

**Protocolo de evaluación:** entrenamos con las primeras N-4 semanas activas y evaluamos en las últimas 4.

In [ ]:
# ── Filtrar al período activo ──────────────────────────────────────────────
MIN_WEEKLY_CLICKS = 50
first_active = weekly[weekly['clicks'] >= MIN_WEEKLY_CLICKS]['week'].min()
weekly_active = weekly[weekly['week'] >= first_active].copy().reset_index(drop=True)

print(f'Semanas totales en el dataset:     {len(weekly)}')
print(f'Primera semana activa (≥{MIN_WEEKLY_CLICKS} clicks): {first_active.date()}')
print(f'Semanas en período activo:         {len(weekly_active)} '
      f'({weekly_active["week"].min().date()} → {weekly_active["week"].max().date()})')
print(f'Clicks en período activo — min:{weekly_active["clicks"].min()}  '
      f'max:{weekly_active["clicks"].max()}  media:{weekly_active["clicks"].mean():.0f}')

# Prophet espera columnas 'ds' (fecha) e 'y' (valor)
df_prophet = weekly_active[['week', 'clicks']].rename(columns={'week': 'ds', 'clicks': 'y'})

# Split temporal: últimas 4 semanas como test
N_TEST = 4
train  = df_prophet.iloc[:-N_TEST].copy()
test   = df_prophet.iloc[-N_TEST:].copy()

print(f'\nTrain: {len(train)} semanas  ({train["ds"].min().date()} → {train["ds"].max().date()})')
print(f'Test:  {len(test)} semanas   ({test["ds"].min().date()} → {test["ds"].max().date()})')

# ── Entrenar Prophet ────────────────────────────────────────────────────────
model = Prophet(
    growth                  = 'linear',
    seasonality_mode        = 'additive',
    weekly_seasonality      = False,   # datos semanales: sin variación intrasemanal
    yearly_seasonality      = False,   # <6 meses activos: patrón anual no estimable
    daily_seasonality       = False,
    interval_width          = 0.90,    # IC 90 % (más conservador con pocos datos)
    changepoint_prior_scale = 0.5,     # flexible para capturar pico + caída
    n_changepoints          = 10,
)
model.fit(train)
print('\nModelo Prophet entrenado.')

In [ ]:
# Predecir sobre las fechas exactas del test (evita desalineamiento con resample)
future_eval = pd.concat([train[['ds']], test[['ds']]], ignore_index=True)
forecast_eval = model.predict(future_eval)

# Las últimas N_TEST filas coinciden con las fechas de test
pred_test = forecast_eval.tail(N_TEST)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)
pred_test['y'] = test['y'].reset_index(drop=True)
pred_test['yhat'] = pred_test['yhat'].clip(lower=0).round(0).astype(int)

# Métricas
mae  = mean_absolute_error(pred_test['y'], pred_test['yhat'])
rmse = mean_squared_error(pred_test['y'], pred_test['yhat']) ** 0.5
mape = ((pred_test['y'] - pred_test['yhat']).abs() / pred_test['y'].replace(0, np.nan)).mean() * 100

print('Evaluación en hold-out (últimas 4 semanas):')
print(f'  MAE:  {mae:.1f} clicks')
print(f'  RMSE: {rmse:.1f} clicks')
print(f'  MAPE: {mape:.1f} %')
print(f'\nPredicción vs real:')
display(pred_test[['ds','y','yhat','yhat_lower','yhat_upper']].reset_index(drop=True))

## 5 · Forecast final — próximas semanas

Re-entrenamos con todos los datos disponibles (train + test) para hacer la predicción
más precisa posible hacia el futuro.

In [ ]:
# Modelo final entrenado con TODOS los datos del período activo (train + test)
model_full = Prophet(
    growth                  = 'linear',
    seasonality_mode        = 'additive',
    weekly_seasonality      = False,
    yearly_seasonality      = False,
    daily_seasonality       = False,
    interval_width          = 0.90,
    changepoint_prior_scale = 0.5,
    n_changepoints          = 10,
)
model_full.fit(df_prophet)

# Predecir FORECAST_WEEKS semanas hacia adelante
future = model_full.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
forecast = model_full.predict(future)
forecast['yhat'] = forecast['yhat'].clip(lower=0)

# Separar histórico de predicción futura
hist_fc = forecast[forecast['ds'] <= df_prophet['ds'].max()]
fut_fc  = forecast[forecast['ds'] >  df_prophet['ds'].max()]

print(f'Forecast para las próximas {FORECAST_WEEKS} semanas:')
display(fut_fc[['ds','yhat','yhat_lower','yhat_upper']]
        .assign(yhat=lambda d: d['yhat'].round(0).astype(int),
                yhat_lower=lambda d: d['yhat_lower'].clip(0).round(0).astype(int),
                yhat_upper=lambda d: d['yhat_upper'].round(0).astype(int))
        .reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

# Datos históricos
ax.plot(df_prophet['ds'], df_prophet['y'],
        color='steelblue', linewidth=1.5, marker='o', markersize=3, label='Clicks reales')

# Ajuste del modelo sobre histórico
ax.plot(hist_fc['ds'], hist_fc['yhat'].clip(0),
        color='tomato', linewidth=1.5, linestyle='--', label='Ajuste del modelo')
ax.fill_between(hist_fc['ds'], hist_fc['yhat_lower'].clip(0), hist_fc['yhat_upper'],
                alpha=0.1, color='tomato')

# Forecast futuro
ax.plot(fut_fc['ds'], fut_fc['yhat'].clip(0),
        color='seagreen', linewidth=2.5, marker='D', markersize=5, label=f'Forecast +{FORECAST_WEEKS} semanas')
ax.fill_between(fut_fc['ds'], fut_fc['yhat_lower'].clip(0), fut_fc['yhat_upper'],
                alpha=0.2, color='seagreen', label='Intervalo de confianza 95 %')

# Línea divisoria histórico / futuro
ax.axvline(df_prophet['ds'].max(), color='gray', linestyle=':', linewidth=1.5)
ax.text(df_prophet['ds'].max(), ax.get_ylim()[1] * 0.95, '  hoy', color='gray', fontsize=9)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.set_ylabel('Clicks / semana')
ax.set_title(f'Forecast de clicks semanales — Prophet  (MAPE holdout: {mape:.1f} %)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Descomposición de los componentes del modelo Prophet
fig = model_full.plot_components(forecast)
fig.suptitle('Componentes del modelo Prophet', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6 · Forecast por sector

Repetimos el forecast para los 3 sectores con más clicks para detectar si alguno
crece o decrece a un ritmo diferente al total.

In [ ]:
top3_sectors = prop[prop['target'] == 1]['sector'].value_counts().head(3).index.tolist()

fig, axes = plt.subplots(len(top3_sectors), 1, figsize=(14, 4 * len(top3_sectors)))
sector_forecasts = {}

for ax, sector in zip(axes, top3_sectors):
    # Serie semanal activa de ese sector (misma lógica de filtrado)
    serie_s = (
        prop[(prop['target'] == 1) & (prop['sector'] == sector)]
        .groupby('week')['id_event'].count()
        .reset_index()
        .rename(columns={'week': 'ds', 'id_event': 'y'})
    )
    serie_s['ds'] = pd.to_datetime(serie_s['ds'])
    # Filtrar al período activo del sector
    first_s = serie_s[serie_s['y'] >= 20]['ds'].min() if (serie_s['y'] >= 20).any() else serie_s['ds'].min()
    serie_s = serie_s[serie_s['ds'] >= first_s].reset_index(drop=True)

    if len(serie_s) < 8:
        ax.set_title(f'{sector} — datos insuficientes ({len(serie_s)} semanas)')
        continue

    m = Prophet(
        weekly_seasonality      = False,
        yearly_seasonality      = False,
        daily_seasonality       = False,
        interval_width          = 0.90,
        changepoint_prior_scale = 0.5,
        n_changepoints          = min(8, len(serie_s) // 3),
    )
    m.fit(serie_s)
    fut_s = m.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
    fc_s  = m.predict(fut_s)
    fc_s['yhat'] = fc_s['yhat'].clip(lower=0)

    hist_s = fc_s[fc_s['ds'] <= serie_s['ds'].max()]
    fut_s_ = fc_s[fc_s['ds'] >  serie_s['ds'].max()]

    ax.plot(serie_s['ds'], serie_s['y'], color='steelblue', linewidth=1.5, marker='o', markersize=3, label='Real')
    ax.plot(hist_s['ds'], hist_s['yhat'], color='tomato', linestyle='--', linewidth=1.2, label='Ajuste')
    ax.plot(fut_s_['ds'], fut_s_['yhat'], color='seagreen', linewidth=2, marker='D', markersize=4, label='Forecast')
    ax.fill_between(fut_s_['ds'], fut_s_['yhat_lower'].clip(0), fut_s_['yhat_upper'],
                    alpha=0.2, color='seagreen')
    ax.axvline(serie_s['ds'].max(), color='gray', linestyle=':', linewidth=1)
    ax.set_title(f'Sector: {sector}  ({len(serie_s)} semanas activas)')
    ax.set_ylabel('Clicks / semana')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.legend(fontsize=8)
    sector_forecasts[sector] = fc_s

plt.suptitle(f'Forecast por sector (top 3) — próximas {FORECAST_WEEKS} semanas', fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Exportación

In [ ]:
# Forecast total — histórico + futuro
df_export = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
df_export.columns = ['week', 'clicks_forecast', 'clicks_lower_95', 'clicks_upper_95']
df_export['clicks_forecast'] = df_export['clicks_forecast'].clip(lower=0).round(0).astype(int)
df_export['clicks_lower_95'] = df_export['clicks_lower_95'].clip(lower=0).round(0).astype(int)
df_export['clicks_upper_95'] = df_export['clicks_upper_95'].round(0).astype(int)
df_export['is_forecast'] = df_export['week'] > df_prophet['ds'].max()

df_export.to_csv(PROCESSED_PATH / 'forecast_clicks.csv', index=False)

print(f'Exportado: forecast_clicks.csv  {df_export.shape}')
print(f'\nForecast futuro:')
display(df_export[df_export['is_forecast']].reset_index(drop=True))
print(f'\nMétricas del modelo (hold-out {N_TEST} semanas):')
print(f'  MAE:  {mae:.1f} clicks/semana')
print(f'  RMSE: {rmse:.1f} clicks/semana')
print(f'  MAPE: {mape:.1f} %')

## Re-auditoría: ¿bate Prophet a baselines triviales? (backtest rolling 1-paso)

Un hold-out de 4 semanas es muy poco para comparar modelos. Hacemos un **backtest rolling de 1 paso**:
para cada semana (desde la 8.ª), entrenamos con todo lo anterior y predecimos la siguiente, comparando
**Prophet** con dos baselines ingenuos: **naïve** (la próxima semana = la última) y **media móvil de 3**.
La diferencia de errores se contrasta con el test de **Diebold-Mariano**.

In [ ]:
# Backtest rolling 1-paso: Prophet vs baselines + Diebold-Mariano
import logging
logging.getLogger("prophet").setLevel(logging.CRITICAL)
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)
from scipy.stats import norm, t

serie = df_prophet.reset_index(drop=True)   # serie del periodo activo (ds, y)
MIN_TRAIN = 8                               # arrancamos con 8 semanas de historia


def prophet_un_paso(train):
    m = Prophet(growth="linear", weekly_seasonality=False, yearly_seasonality=False,
                daily_seasonality=False, changepoint_prior_scale=0.5,
                n_changepoints=min(10, len(train) - 1))
    m.fit(train)
    futuro = m.make_future_dataframe(periods=1, freq="W-MON")
    return max(0.0, m.predict(futuro)["yhat"].iloc[-1])


errores = {"Prophet": [], "Naive": [], "Media movil 3": []}
for i in range(MIN_TRAIN, len(serie)):
    train = serie.iloc[:i]
    real = serie["y"].iloc[i]
    errores["Prophet"].append(real - prophet_un_paso(train))
    errores["Naive"].append(real - train["y"].iloc[-1])
    errores["Media movil 3"].append(real - train["y"].iloc[-3:].mean())
for k in errores:
    errores[k] = np.array(errores[k])

print("Backtest rolling 1-paso (", len(errores["Prophet"]), "origenes):")
print()
print(f"{'Modelo':<16}{'MAE':>9}{'RMSE':>9}")
for nombre, e in errores.items():
    print(f"{nombre:<16}{np.abs(e).mean():>9.1f}{np.sqrt((e ** 2).mean()):>9.1f}")


def diebold_mariano(e1, e2, h=1):
    """DM con correccion de Harvey-Leybourne-Newbold (HLN) para muestras pequenas:
    ajusta el estadistico por el tamano muestral y usa la t de Student con n-1 g.l.
    (en vez de la normal), que es lo adecuado con pocos origenes (aqui n=14)."""
    d = e1 ** 2 - e2 ** 2
    n = len(d)
    var = d.var(ddof=1) / n
    if var <= 0:
        return np.nan, np.nan
    stat = d.mean() / np.sqrt(var)
    # Factor HLN (horizonte h): sqrt((n + 1 - 2h + h(h-1)/n) / n); para h=1 -> sqrt((n-1)/n)
    factor = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat_hln = stat * factor
    p = 2 * (1 - t.cdf(abs(stat_hln), df=n - 1))
    return stat_hln, p


print()
print("Test de Diebold-Mariano con correccion HLN (muestra pequena, n=14; stat>0 = baseline mejor):")
for nombre in ["Naive", "Media movil 3"]:
    stat, p = diebold_mariano(errores["Prophet"], errores[nombre])
    print(f"  Prophet vs {nombre:<14} stat={stat:+.3f}  p={p:.3f}")
print()
print("Conclusion: en esta serie corta y volatil, los baselines triviales igualan o superan a Prophet.")